# Sawti-ASR — analyse des écritures des transcriptions (CPU)

Régénère `data/processed` depuis `main`, puis compte par split les extraits : (a) arabe seul, (b) arabe + latin, (c) latin seul, (d) vides, avec 5 exemples par catégorie.
Les jetons sont lus dans le dataset Kaggle PRIVÉ `sawti-secrets` ; aucune valeur n'est affichée.

In [ ]:
REPO_URL = "https://github.com/Mohamedaminebelasri/sawti-asr"
BRANCHE = "main"

In [ ]:
import glob, os

# Les jetons viennent du dataset PRIVÉ sawti-secrets (fichier secrets.env, lignes NOM=valeur),
# attaché par dataset_sources dans kernel-metadata.json. On n'affiche JAMAIS une valeur.
SECRETS = ["HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]
fichiers = sorted(glob.glob("/kaggle/input/**/secrets.env", recursive=True))
print("Fichier de jetons trouvé :", "oui" if fichiers else "non")
lus = {}
if fichiers:
    with open(fichiers[0], encoding="utf-8-sig") as f:
        for ligne in f:
            ligne = ligne.strip()
            if not ligne or ligne.startswith("#") or "=" not in ligne:
                continue
            nom, valeur = ligne.split("=", 1)
            lus[nom.strip()] = valeur.strip().strip('"').strip("'")
for nom in SECRETS:
    if lus.get(nom):
        os.environ[nom] = lus[nom]
        print(f"SECRET {nom}: OK")
    else:
        print(f"SECRET {nom}: ABSENT")
del lus
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

In [ ]:
import subprocess, sys, time

TRAVAIL = "/kaggle/temp/sawti-asr" if os.path.isdir("/kaggle") else os.path.abspath("sawti-asr")
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("sortie")
os.makedirs(SORTIE, exist_ok=True)
JOURNAL = os.path.join(SORTIE, "run_log.txt")

def sh(cmd, cwd=None):
    """Lance une commande, affiche et journalise sa sortie, s'arrête si elle échoue."""
    debut = time.time()
    print(f"\n$ {cmd}", flush=True)
    with open(JOURNAL, "a", encoding="utf-8") as j:
        j.write(f"\n$ {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="", flush=True)
            j.write(ligne)
        p.wait()
        duree = time.time() - debut
        j.write(f"[code retour {p.returncode}, {duree:.0f} s]\n")
    print(f"[code retour {p.returncode}, {duree:.0f} s]", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"Échec ({p.returncode}) : {cmd}")

sh(f"mkdir -p {os.path.dirname(TRAVAIL)} && rm -rf {TRAVAIL} && git clone -q --depth 1 -b {BRANCHE} {REPO_URL} {TRAVAIL}")
sh("git log -1 --format='commit %h %s'", cwd=TRAVAIL)
sh(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=TRAVAIL)
sh(f"{sys.executable} -m pip install -q -e .", cwd=TRAVAIL)

## Préparation des données (CPU)

In [ ]:
sh("python scripts/prepare_data.py", cwd=TRAVAIL)

## Analyse

In [ ]:
# Analyse des écritures dans data/processed (colonne texte seulement, pas d'audio).
import json, re
from datasets import load_from_disk

AR = re.compile(r"[؀-ۿݐ-ݿ]")
LAT = re.compile(r"[A-Za-zÀ-ɏ]")  # même plage que has_latin

def categorie(t):
    t = (t or "").strip()
    if not t:
        return "d_vide"
    a, l = bool(AR.search(t)), bool(LAT.search(t))
    if a and l:
        return "b_arabe_latin"
    if a:
        return "a_arabe_seul"
    if l:
        return "c_latin_seul"
    return "e_sans_lettre"  # chiffres ou symboles seulement

CATS = ["a_arabe_seul", "b_arabe_latin", "c_latin_seul", "d_vide", "e_sans_lettre"]
dd = load_from_disk(os.path.join(TRAVAIL, "data/processed"))
rapport = {}
for split, ds in dd.items():
    textes = ds["text"]
    sources = ds["source"]
    comptes = {c: 0 for c in CATS}
    par_source = {}
    exemples = {c: [] for c in CATS}
    for t, s in zip(textes, sources):
        c = categorie(t)
        comptes[c] += 1
        par_source.setdefault(s, {k: 0 for k in CATS})[c] += 1
        if len(exemples[c]) < 5:
            exemples[c].append(t)
    rapport[split] = {"n": len(textes), "comptes": comptes,
                      "par_source": par_source, "exemples": exemples}

lignes = ["| split | total | (a) arabe seul | (b) arabe + latin | (c) latin seul | (d) vides | (e) sans lettre |",
          "|---|---|---|---|---|---|---|"]
for split, r in rapport.items():
    n = max(r["n"], 1)
    cel = [f'{r["comptes"][c]} ({100 * r["comptes"][c] / n:.1f} %)' for c in CATS]
    lignes.append(f"| {split} | {r['n']} | " + " | ".join(cel) + " |")
tableau = "\n".join(lignes)
print(tableau)
for split, r in rapport.items():
    print(f"\n=== {split} — par source : {r['par_source']}")
    for c in CATS:
        if r["exemples"][c]:
            print(f"  [{c}]")
            for e in r["exemples"][c]:
                print("    ·", repr(e))

with open(os.path.join(SORTIE, "analyse_transcriptions.json"), "w", encoding="utf-8") as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)
with open(os.path.join(SORTIE, "analyse_transcriptions.md"), "w", encoding="utf-8") as f:
    f.write(tableau + "\n")
print("\nTERMINE : analyse écrite dans", SORTIE)
